<a href="https://colab.research.google.com/github/RadhikaDeshpande1010/pyspark-rdd-sales-analytics/blob/main/pyspark_rdd_sales_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark RDD — Sales Data Analysis

This notebook demonstrates core PySpark RDD operations — `map()`, `filter()`, and `reduceByKey()` — applied to a retail sales dataset.  
Each section addresses a specific analytical question using a functional transformation pipeline.

**Dataset schema:** `order_id, customer_id, product, category, city, quantity, unit_price`

---
## Setup — Initialize Spark Session and Context

In [1]:
from pyspark.sql import SparkSession

spark_session = SparkSession.builder \
    .master("local") \
    .appName("PySpark RDD Sales Analysis") \
    .getOrCreate()

print(spark_session)

In [2]:
spark_context = spark_session.sparkContext
print(spark_context)

<SparkContext master=local appName=PySpark RDD Sales Analysis>


---
## Data Ingestion

In [3]:
# Load raw sales data from text file into an RDD
raw_sales_rdd = spark_context.textFile("/content/sample_data/sales_data.txt")

---
## Q1 — Parse Records into Structured Lists
Load the `sales_data.txt` file into an RDD and convert each record into a list by splitting on commas.

In [4]:
# Split each raw line on commas to produce a list of fields per record
parsed_sales_rdd = raw_sales_rdd.map(lambda raw_line: raw_line.strip().split(","))

parsed_sales_rdd.take(5)

[['1', 'C112', 'Printer', 'Electronics', 'Chennai', '5', '12000'],
 ['2', 'C137', 'Headphones', 'Electronics', 'Delhi', '5', '2000'],
 ['3', 'C150', 'Phone', 'Electronics', 'Patna', '1', '30000'],
 ['4', 'C113', 'Tablet', 'Electronics', 'Hyderabad', '5', '25000'],
 ['5', 'C150', 'Bed', 'Furniture', 'Hyderabad', '4', '18000']]

---
## Q2 — Total Price per Order
Using `map()`, calculate the total price for each record (`quantity × unit_price`).  
Output: `(order_id, total_price)`.

In [5]:
# Compute total revenue per order: quantity (index 5) * unit_price (index 6)
order_total_price_rdd = parsed_sales_rdd.map(
    lambda record: (record[0], int(record[5]) * int(record[6]))
)

print(order_total_price_rdd.collect())

[('1', 60000), ('2', 10000), ('3', 30000), ('4', 125000), ('5', 72000), ('6', 36000), ('7', 14000), ('8', 110000), ('9', 8000), ('10', 150000), ('11', 2000), ('12', 25000), ('13', 120000), ('14', 21000), ('15', 24000), ('16', 36000), ('17', 54000), ('18', 8000), ('19', 100000), ('20', 72000), ('21', 25000), ('22', 8000), ('23', 48000), ('24', 16000), ('25', 125000), ('26', 100000), ('27', 36000), ('28', 15000), ('29', 125000), ('30', 90000), ('31', 60000), ('32', 14000), ('33', 45000), ('34', 10000), ('35', 60000), ('36', 6000), ('37', 30000), ('38', 12000), ('39', 54000), ('40', 10000), ('41', 15000), ('42', 12000), ('43', 7000), ('44', 80000), ('45', 100000), ('46', 21000), ('47', 275000), ('48', 100000), ('49', 15000), ('50', 20000), ('51', 30000), ('52', 60000), ('53', 28000), ('54', 50000), ('55', 220000), ('56', 48000), ('57', 18000), ('58', 165000), ('59', 72000), ('60', 2000), ('61', 25000), ('62', 90000), ('63', 120000), ('64', 60000), ('65', 60000), ('66', 125000), ('67', 250

---
## Q3 — Electronics Orders
Using `filter()`, extract the `order_id` for all orders where the category is **Electronics**.

In [6]:
# Filter records where category is Electronics
electronics_category_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], record[3])) \
    .filter(lambda order_category: order_category[1] == "Electronics") \
    .map(lambda order_category: order_category[0])

print(electronics_category_orders_rdd.collect())

['1', '2', '3', '4', '8', '10', '11', '12', '13', '15', '16', '18', '19', '21', '22', '23', '25', '26', '27', '28', '29', '30', '31', '33', '34', '35', '36', '37', '38', '40', '41', '42', '45', '47', '48', '49', '51', '54', '55', '56', '58', '60', '61', '63', '65', '66', '67', '69', '70', '71', '72', '73', '74', '77', '78', '79', '80', '81', '82', '87', '88', '89', '90', '95', '96', '97']


---
## Q4 — Orders with Quantity Greater Than 2
Using `filter()`, find all records where the quantity sold is greater than 2.

In [7]:
# Filter records where quantity (index 5) > 2
high_quantity_orders_rdd = parsed_sales_rdd.filter(
    lambda record: int(record[5]) > 2
)

high_quantity_orders_rdd.count()

60

---
## Q5 — Total Sales Amount per Product
Using `map()` and `reduceByKey()`, calculate the total sales amount for each product.

In [8]:
# Key by product name (index 2), value = quantity * unit_price
total_sales_per_product_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda sales_a, sales_b: sales_a + sales_b)

print(total_sales_per_product_rdd.collect())

[('Printer', 408000), ('Headphones', 58000), ('Phone', 1260000), ('Tablet', 1350000), ('Bed', 666000), ('Table', 161000), ('Laptop', 880000), ('Chair', 52000), ('Monitor', 390000), ('Sofa', 380000)]


---
## Q6 — Total Quantity Sold per Product
Using `map()` and `reduceByKey()`, calculate the total quantity sold for each product.

In [9]:
# Key by product name (index 2), value = quantity (index 5)
total_quantity_per_product_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], int(record[5]))) \
    .reduceByKey(lambda qty_a, qty_b: qty_a + qty_b)

print(total_quantity_per_product_rdd.collect())

[('Printer', 34), ('Headphones', 29), ('Phone', 42), ('Tablet', 54), ('Bed', 37), ('Table', 23), ('Laptop', 16), ('Chair', 13), ('Monitor', 26), ('Sofa', 19)]


---
## Q7 — Orders with Unit Price Greater Than 20,000
Using `filter()`, find all orders where the unit price exceeds 20,000.

In [10]:
# Map to (order_id, unit_price) then filter on unit_price > 20000
high_unit_price_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], int(record[6]))) \
    .filter(lambda order_price: order_price[1] > 20000)

print(high_unit_price_orders_rdd.collect())

[('3', 30000), ('4', 25000), ('8', 55000), ('10', 30000), ('12', 25000), ('13', 30000), ('19', 25000), ('21', 25000), ('25', 25000), ('26', 25000), ('29', 25000), ('30', 30000), ('37', 30000), ('45', 25000), ('47', 55000), ('48', 25000), ('54', 25000), ('55', 55000), ('58', 55000), ('61', 25000), ('63', 30000), ('66', 25000), ('67', 25000), ('69', 30000), ('70', 25000), ('77', 30000), ('78', 30000), ('80', 55000), ('82', 30000), ('87', 30000), ('88', 25000), ('89', 25000), ('96', 25000), ('97', 30000)]


---
## Q8 — Total Revenue per City
Using `map()` and `reduceByKey()`, calculate the total sales **revenue** (`quantity × unit_price`) for each city.


In [11]:
# Key by city (index 4), value = quantity * unit_price
total_revenue_per_city_rdd = parsed_sales_rdd \
    .map(lambda record: (record[4], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_city_rdd.collect())

[('Chennai', 687000), ('Delhi', 712000), ('Patna', 294000), ('Hyderabad', 597000), ('Mumbai', 903000), ('Bangalore', 1001000), ('Pune', 1157000), ('Kolkata', 254000)]


---
## Q9 — Furniture Orders with Quantity Greater Than 1
Using `filter()`, extract all orders where the category is **Furniture** and the quantity is greater than 1.

In [12]:
# Map to (order_id, category, quantity), filter on Furniture + qty > 1, then project to (order_id, quantity)
furniture_bulk_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], record[3], int(record[5]))) \
    .filter(lambda order_cat_qty: order_cat_qty[1] == 'Furniture' and order_cat_qty[2] > 1) \
    .map(lambda order_cat_qty: (order_cat_qty[0], order_cat_qty[2]))

print(furniture_bulk_orders_rdd.collect())

[('5', 4), ('6', 2), ('7', 2), ('9', 2), ('14', 3), ('17', 3), ('20', 4), ('24', 4), ('32', 2), ('39', 3), ('44', 4), ('46', 3), ('52', 3), ('53', 4), ('59', 4), ('62', 5), ('64', 3), ('68', 3), ('76', 3), ('84', 4), ('85', 4), ('86', 4), ('91', 3), ('92', 2), ('94', 3), ('98', 2), ('99', 2)]


---
## Q10 — Order Count per Category
Using `map()` and `reduceByKey()`, count the total number of orders for each category.

In [13]:
# Key by category (index 3), value = 1 per order, then sum
order_count_per_category_rdd = parsed_sales_rdd \
    .map(lambda record: (record[3], 1)) \
    .reduceByKey(lambda count_a, count_b: count_a + count_b)

print(order_count_per_category_rdd.collect())

[('Electronics', 66), ('Furniture', 34)]


---
## Q11 — Total Revenue per Customer
Using `map()` and `reduceByKey()`, calculate the total revenue generated by each customer.


In [14]:
# Key by customer_id (index 1), value = quantity * unit_price
total_revenue_per_customer_rdd = parsed_sales_rdd \
    .map(lambda record: (record[1], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_customer_rdd.collect())

[('C112', 160000), ('C137', 10000), ('C150', 389000), ('C113', 133000), ('C103', 177000), ('C144', 39000), ('C107', 384000), ('C125', 224000), ('C124', 310000), ('C128', 27000), ('C118', 25000), ('C102', 161000), ('C119', 263000), ('C108', 48000), ('C114', 36000), ('C105', 54000), ('C120', 86000), ('C106', 25000), ('C109', 30000), ('C122', 60000), ('C141', 96000), ('C134', 480000), ('C147', 325000), ('C143', 79000), ('C123', 170000), ('C111', 60000), ('C129', 106000), ('C148', 48000), ('C110', 204000), ('C116', 85000), ('C140', 308000), ('C135', 244000), ('C133', 22000), ('C145', 153000), ('C101', 168000), ('C138', 7000), ('C132', 60000), ('C127', 140000), ('C149', 72000), ('C100', 16000), ('C136', 25000), ('C115', 60000), ('C130', 36000)]


---
## Q12 — Distinct Products Sold in Delhi
Using `filter()` and `map()`, find all unique products sold in Delhi.

In [15]:
# Filter records for Delhi (index 4), extract product name (index 2), deduplicate
distinct_products_in_delhi_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], record[4])) \
    .filter(lambda product_city: product_city[1] == "Delhi") \
    .map(lambda product_city: product_city[0]) \
    .distinct()

print(distinct_products_in_delhi_rdd.collect())

['Headphones', 'Chair', 'Table', 'Tablet', 'Printer', 'Phone', 'Monitor', 'Laptop', 'Bed']


---
## Q13 — Total Quantity Sold per City
Using `map()` and `reduceByKey()`, find the total quantity sold in each city.

In [16]:
# Key by city (index 4), value = quantity (index 5)
total_quantity_per_city_rdd = parsed_sales_rdd \
    .map(lambda record: (record[4], int(record[5]))) \
    .reduceByKey(lambda qty_a, qty_b: qty_a + qty_b)

print(total_quantity_per_city_rdd.collect())

[('Chennai', 33), ('Delhi', 44), ('Patna', 19), ('Hyderabad', 34), ('Mumbai', 47), ('Bangalore', 49), ('Pune', 45), ('Kolkata', 22)]


---
## Q14 — Orders with Total Sales Value Greater Than 50,000
Using `filter()`, extract all orders where the total sales value (`quantity × unit_price`) exceeds 50,000.

In [17]:
# Map to (order_id, total_sales), filter where total_sales > 50000
high_value_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], int(record[5]) * int(record[6]))) \
    .filter(lambda order_sales: order_sales[1] > 50000)

print(high_value_orders_rdd.collect())

[('1', 60000), ('4', 125000), ('5', 72000), ('8', 110000), ('10', 150000), ('13', 120000), ('17', 54000), ('19', 100000), ('20', 72000), ('25', 125000), ('26', 100000), ('29', 125000), ('30', 90000), ('31', 60000), ('35', 60000), ('39', 54000), ('44', 80000), ('45', 100000), ('47', 275000), ('48', 100000), ('52', 60000), ('55', 220000), ('58', 165000), ('59', 72000), ('62', 90000), ('63', 120000), ('64', 60000), ('65', 60000), ('66', 125000), ('69', 150000), ('76', 60000), ('77', 120000), ('78', 150000), ('79', 60000), ('80', 110000), ('82', 120000), ('84', 80000), ('85', 72000), ('87', 60000), ('88', 100000), ('94', 54000), ('95', 60000), ('96', 125000), ('97', 120000)]


---
## Q15 — Total Revenue per Category
Using `map()` and `reduceByKey()`, calculate the total revenue generated for each category.

In [18]:
# Key by category (index 3), value = quantity * unit_price
total_revenue_per_category_rdd = parsed_sales_rdd \
    .map(lambda record: (record[3], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_category_rdd.collect())

[('Electronics', 4346000), ('Furniture', 1259000)]
